[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ofermend/hands-on-rag/blob/main/chapter8/simple-table-chunking.ipynb)

# Chapter 8: Table Extraction and Chunking

Tables pose a unique challenge for RAG chunking: naive text splitting breaks rows across chunks, destroying the column-header relationships that give tabular data its meaning. A single product row split from its header becomes meaningless. This notebook demonstrates the problem by showing what happens when standard text splitters are applied to Markdown tables.

This notebook shows table-to-text conversion strategies and illustrates chunking pitfalls with tabular data.

**What you'll learn:**
- Convert DataFrames with multi-level columns to Markdown and JSON formats
- Observe how standard text splitting destroys table structure
- Understand why tables need specialized chunking strategies

**Prerequisites:** `pip install langchain-text-splitters pandas`

In [20]:
import json
from typing import Final

import pandas
from langchain_text_splitters import RecursiveCharacterTextSplitter

## Create Sample Table

We build a product pricing DataFrame with multi-level column headers to simulate the kind of structured tables commonly found in business documents.

In [ ]:
product_data_rows: list[list[str]] = [
    ["Model-A", "$299", "€257.25", "4.5 Stars"],
    ["Model-B", "$449", "€386.95", "4.2 Stars"],
    ["Model-C", "$199", "€171.50", "3.9 Stars"],
    ["Model-X", "$899", "€774.77", "4.8 Stars"],
    ["Model-Pro", "$1299", "€1122.15", "4.9 Stars"],
    ["Model-Lite", "$149", "€128.70", "3.5 Stars"],
    ["Model-Air", "$699", "€603.75", "4.4 Stars"],
    ["Model-Eco", "$99", "€85.52", "3.1 Stars"],
    ["Model-Max", "$1099", "€949.20", "4.7 Stars"],
    ["Model-Std", "$249", "€215.04", "4.0 Stars"],
    ["Model-Ultra", "$1599", "€1381.13", "4.8 Stars"],
    ["Model-Mini", "$349", "€301.44", "4.1 Stars"],
    ["Model-Go", "$179", "€154.60", "3.7 Stars"],
    ["Model-Titan", "$1999", "€1726.63", "4.9 Stars"],
]

# Each tuple is (top-level header, sub-level header).
# An empty string marks a column that has no sub-header.
column_header_pairs: list[tuple[str, str]] = [
    ("Product", ""),
    ("Price", "USD"),
    ("Price", "EUR"),
    ("Rating", ""),
]

# pandas ships no type stubs here, so the pandas annotations in this notebook
# document the expected types; the type checker does not verify them.
multi_level_columns: pandas.MultiIndex = pandas.MultiIndex.from_tuples(tuples=column_header_pairs)

product_dataframe: pandas.DataFrame = pandas.DataFrame(
    data=product_data_rows,
    columns=multi_level_columns,
)

In [ ]:
product_table_text: str = product_dataframe.to_string()

print(product_table_text)

## Convert Table Formats

We build the JSON and Markdown text by hand, with plain loops over the DataFrame's columns and rows, so every step of the conversion is visible. Each two-level header such as `("Price", "USD")` becomes one flat label such as `Price USD`.

In [ ]:
TOP_HEADER_LEVEL: Final[int] = 0
SUB_HEADER_LEVEL: Final[int] = 1


def build_column_label(
    *,
    top_level_header: str,
    sub_level_header: str,
) -> str:
    """Join a top-level header and a sub-level header into one label, such as "Price USD"."""
    if sub_level_header == "":
        return top_level_header

    column_label = f"{top_level_header} {sub_level_header}"
    return column_label


def build_column_labels(dataframe: pandas.DataFrame) -> list[str]:
    """Return one flat label per column, in column order, such as ["Product", "Price USD", ...]."""
    column_count = len(dataframe.columns)

    # Each header level is read on its own, so every header is a plain string.
    dataframe_columns = dataframe.columns
    top_level_headers = dataframe_columns.get_level_values(level=TOP_HEADER_LEVEL)
    sub_level_headers = dataframe_columns.get_level_values(level=SUB_HEADER_LEVEL)

    column_labels: list[str] = []

    for column_position in range(column_count):
        top_level_header_value = top_level_headers[column_position]
        top_level_header = str(top_level_header_value)

        sub_level_header_value = sub_level_headers[column_position]
        sub_level_header = str(sub_level_header_value)

        column_label = build_column_label(
            top_level_header=top_level_header,
            sub_level_header=sub_level_header,
        )

        column_labels.append(column_label)

    return column_labels

In [ ]:
PREVIEW_ROW_COUNT: Final[int] = 4

preview_dataframe: pandas.DataFrame = product_dataframe.head(n=PREVIEW_ROW_COUNT)

preview_column_count = len(preview_dataframe.columns)
preview_row_count = len(preview_dataframe.index)

preview_column_labels = build_column_labels(preview_dataframe)

# Same shape as pandas orient="columns": {column_label: {row_label: cell_value}}.
preview_cells_by_column: dict[str, dict[str, str]] = {}

for column_position in range(preview_column_count):
    column_label = preview_column_labels[column_position]

    cells_by_row_label: dict[str, str] = {}

    for row_position in range(preview_row_count):
        row_label_value = preview_dataframe.index[row_position]
        row_label = str(row_label_value)

        cell_value = preview_dataframe.iat[row_position, column_position]
        cell_text = str(cell_value)

        cells_by_row_label[row_label] = cell_text

    preview_cells_by_column[column_label] = cells_by_row_label

# ensure_ascii=False writes "€" as-is, instead of as an ASCII escape code.
preview_json_text = json.dumps(
    preview_cells_by_column,
    ensure_ascii=False,
    indent=2,
)

print(preview_json_text)

In [ ]:
ROW_LABEL_HEADER: Final[str] = "Row"

table_column_count = len(product_dataframe.columns)
table_row_count = len(product_dataframe.index)

table_column_labels = build_column_labels(product_dataframe)

markdown_lines: list[str] = []

# Header line, for example: | Row | Product | Price USD | Price EUR | Rating |
header_line = "| " + ROW_LABEL_HEADER + " |"

for table_column_label in table_column_labels:
    header_line = header_line + " " + table_column_label + " |"

markdown_lines.append(header_line)

# Separator line: one "---" for the row label column and one for each data column.
separator_line = "|---|"

for _column_position in range(table_column_count):
    separator_line = separator_line + "---|"

markdown_lines.append(separator_line)

# One line per row, for example: | 0 | Model-A | $299 | €257.25 | 4.5 Stars |
for row_position in range(table_row_count):
    table_row_label_value = product_dataframe.index[row_position]
    table_row_label = str(table_row_label_value)

    row_line = "| " + table_row_label + " |"

    for column_position in range(table_column_count):
        table_cell_value = product_dataframe.iat[row_position, column_position]
        table_cell_text = str(table_cell_value)

        row_line = row_line + " " + table_cell_text + " |"

    markdown_lines.append(row_line)

markdown_table_text = "\n".join(markdown_lines)

print(markdown_table_text)

## Demonstrate Chunking

Using a deliberately small chunk size, we show how standard text splitting tears apart table rows from their headers, destroying the structured relationships that give the data meaning.

In [ ]:
# The chunk size is intentionally small so the split breaks table rows apart.
CHUNK_SIZE_CHARACTERS: Final[int] = 100
CHUNK_OVERLAP_CHARACTERS: Final[int] = 20

# These are the library's default separators, written out so the split order is visible:
# paragraph break, then line break, then space, then individual characters.
chunk_separators: list[str] = ["\n\n", "\n", " ", ""]

text_splitter = RecursiveCharacterTextSplitter(
    separators=chunk_separators,
    keep_separator=True,
    is_separator_regex=False,
    chunk_size=CHUNK_SIZE_CHARACTERS,
    chunk_overlap=CHUNK_OVERLAP_CHARACTERS,
    length_function=len,
    strip_whitespace=True,
)

table_chunks: list[str] = text_splitter.split_text(markdown_table_text)

chunk_count = len(table_chunks)

print(f"\n--- SPLIT INTO {chunk_count} CHUNKS (Chunk Size: {CHUNK_SIZE_CHARACTERS}) ---")

for chunk_index in range(chunk_count):
    chunk_text = table_chunks[chunk_index]
    chunk_number = chunk_index + 1

    # repr() shows the newline characters (\n) inside each chunk.
    chunk_representation = repr(chunk_text)

    print(f"\n[CHUNK {chunk_number}]")
    print(chunk_representation)